In [ ]:
import itertools
from dataclasses import asdict
from pathlib import Path

import pandas as pd

from arms import FILTERS, FilterParams, HeadSpec, build_correlator, estimator_factory
from uavloc.simulation.sequence import flight_metrics, load_flight_set, per_flight_table, run_flights
from uavloc.track import experiment_dir, set_seed

# Params

In [ ]:
EXP = "027-sequence-filters"
SEED = 42
DEVICE = "cpu"                    # CPU-only process: MPS corrupts CPU conv2d in the same process (bug-121)
ROUTE_SET = "sim-flights"
INITIAL_ERROR_M = {"normal": 0.0, "cold": 40.0}
SIGMA0_M = {"normal": 1.0, "cold": 25.0}
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp",
    "seg_checkpoint_drone": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_checkpoint_sat": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_channels": "all", "drone_seg_gsd_m": 0.385, "match_gsd_m": 0.385, "matching": "weighted-ncc",
    "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5], "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0,
    "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
N_PARTICLES = 2000
K_MAX_HYPOTHESES = 5
PEAKS_PER_HYPOTHESIS = 3
PEAK_NMS_M = 10.0
MERGE_DISTANCE_M = 5.0
CHILD_FLOOR_SIGMA_M = 1.0
WINDOW_SIGMAS = 3.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
HEAD_RUN = None                   # e.g. "023.3": score with that 023 correlation head instead of weighted NCC
HEAD_EXP = "023-learned-correlation-head"
HEAD_ENCODER_CHANNELS = [32, 32, 16]
HEAD_REFINE_CHANNELS = 16
HEAD_CLEAN_VILLAGES = ["bushcha-50p2422-25p8305", "ivannie-50p4543-25p7398", "maidan-50p2512-26p0155",
                       "maksymivka-48p8871-23p9317", "oblaznytsia-49p2703-24p1288", "pidhaichyky-48p5627-25p1788",
                       "pidhorodnie-49p5387-25p5199", "sadzhavka-48p5678-24p7897", "strilche-48p6821-25p5613",
                       "vesnianka-49p7175-27p3183", "zlynets-50p3697-25p7999"]   # villages the 023 head never trained on
TUNING_DIR = "artifacts/tuning"
SCENARIO = "normal"
SPLIT = "tuning"
TEMPERATURES = [0.25, 0.5, 1.0, 2.0]
Q_POS_M_PER_SQRT_S = [0.1, 0.3, 1.0, 2.0]
FIXED_Q = None                    # e.g. {"particles": 2.0, "hypotheses": 1.0}: tune T only, at these q
SHARD = 0
N_SHARDS = 4

In [ ]:
set_seed(SEED)

# Data

In [ ]:
set_config, specs = load_flight_set(ROUTE_SET, split=SPLIT)
len(specs)

# Model

In [ ]:
def filter_params(temperature: float, q_pos: float) -> FilterParams:
    return FilterParams(temperature, q_pos, N_PARTICLES, K_MAX_HYPOTHESES, PEAKS_PER_HYPOTHESIS, PEAK_NMS_M,
                        MERGE_DISTANCE_M, CHILD_FLOOR_SIGMA_M, WINDOW_SIGMAS, WINDOW_CAP_HALF_M, WINDOW_MARGIN_M)

In [ ]:
def correlator_for(map_raster):
    head = None if HEAD_RUN is None else HeadSpec(experiment_dir(HEAD_EXP) / "artifacts" / f"head-{HEAD_RUN}.pt",
                                                  tuple(HEAD_ENCODER_CHANNELS), HEAD_REFINE_CHANNELS)
    return build_correlator(map_raster, CORRELATOR, DEVICE, head)

In [ ]:
grid = [(arm, t, q) for arm, t, q in itertools.product(FILTERS, TEMPERATURES, Q_POS_M_PER_SQRT_S)
        if FIXED_Q is None or q == FIXED_Q[arm]]
combos = grid[SHARD::N_SHARDS]
factories = {f"{arm}|{t}|{q}": estimator_factory(arm, filter_params(t, q)) for arm, t, q in combos}
list(factories)

# Run

In [ ]:
outcomes = run_flights(set_config, specs, INITIAL_ERROR_M[SCENARIO], SIGMA0_M[SCENARIO], correlator_for, factories)

In [ ]:
summary = pd.DataFrame([{"filter": name.split("|")[0], "temperature": float(name.split("|")[1]),
                         "q_pos_m_per_sqrt_s": float(name.split("|")[2]), **flight_metrics(found)}
                        for name, found in outcomes.items()])
summary.sort_values(["filter", "mean_end_error_m"])

In [ ]:
flights = pd.concat([per_flight_table(found).assign(combo=name) for name, found in outcomes.items()], ignore_index=True)
out = Path(TUNING_DIR)
out.mkdir(parents=True, exist_ok=True)
summary.to_csv(out / f"shard-{SHARD}.csv", index=False)
flights.to_csv(out / f"shard-{SHARD}-flights.csv", index=False)